<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-16 15:35:11
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.53 C
CAGR/XIRR %:     1.16%
Est FTT Value:   2.74 C
Future Upside:   1.20 C (78.55%)
=================== PERFORMANCE ==========================
Current PnL:             -41.97 L (-21.87%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    7.25 L (4.87%)
CY Booked PnL:           6.36 L (4.15%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               -45.37 K (-0.3%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.46 L
Total Loss:               -44.44 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-0.28,-26.76,36.67,0.09,27064.0,-26972.0,73805.0,770.00,-10.92,55.0,M-SC,1.98,250.0,-1.00,0.48,32.84,OX40N,NTT,DURABLES
12,BOSCH-HCIL,-3.18,-24.78,32.92,-0.01,31650.0,-31664.0,96141.0,2282.00,750.61,47.0,M-SC,19.43,231.0,-1.00,0.63,63.65,OX40N,BTT,AC
76,UNITDSPR,2.06,6.22,13.69,20.77,34049.0,14568.0,248712.0,1644.00,-9.49,61.0,X-MC,1.88,64.0,0.43,1.62,19.55,X40N,NTT,BREWERIES
65,SIS,-1.82,-12.23,38.17,21.26,37196.0,-13584.0,97448.0,528.00,87.50,44.0,H-SC,6.33,168.0,-0.37,0.64,44.89,OX40N,NTT,MISC
52,MASFIN,-0.09,-18.65,49.05,21.26,39098.0,-18270.0,79710.0,396.02,-37.37,51.0,H-SC,14.76,167.0,-0.47,0.52,8.60,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,0.39,6.56,33.53,42.29,101321.0,18609.0,302180.0,2782.73,11.66,56.0,X-LC,2.65,29.0,0.18,1.97,24.39,X40N,ATH,AUTO
73,TMPV,-8.70,-34.57,128.66,49.60,401411.0,-164878.0,311994.0,600.00,-87.07,37.0,X-LC,16.42,4.0,-0.41,2.04,0.46,XY24,NTT,AUTO
51,MARUTI,-2.20,-7.49,36.19,25.99,113736.0,-25435.0,314275.0,17120.06,-13.78,59.0,X-LC,9.46,33.0,-0.22,2.05,12.54,XR,ATH,AUTO
47,JIOFIN,-0.66,-20.53,71.24,36.09,227335.0,-82433.0,319112.0,387.00,-39.12,57.0,X-LC,8.77,34.0,-0.36,2.08,10.51,XY24,BTT,FINANCE
44,INFY,0.32,-17.16,72.21,42.66,239794.0,-68776.0,332079.0,1972.00,-40.08,56.0,X-LC,7.20,3.0,-0.29,2.17,18.88,X40,NTT,IT


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,3.54,-15.97,53.17,28.70,195503.0,-69900.0,367695.0,452.00,-24.24,73.0,X-LC,6.78,1.0,-0.36,2.40,15.73,X40,NTT,FMCG
53,NESTLEIND,2.68,8.83,18.74,29.22,63681.0,27576.0,339814.0,1540.00,-17.76,55.0,X-LC,9.18,20.0,0.43,2.22,11.69,XY25,ATH,FMCG
28,GODREJCP,2.52,-18.79,62.67,32.10,126996.0,-46894.0,202643.0,1452.16,-20.39,63.0,X-MC,15.47,32.0,-0.37,1.32,9.79,X40N,ATH,FMCG
35,HDFCLIFE,2.07,4.92,34.23,40.83,123945.0,16964.0,362096.0,811.41,-23.81,74.0,X-LC,8.32,31.0,0.14,2.36,17.70,X40,ATH,INSURANCE
76,UNITDSPR,2.06,6.22,13.69,20.77,34049.0,14568.0,248712.0,1644.00,-9.49,61.0,X-MC,1.88,64.0,0.43,1.62,19.55,X40N,NTT,BREWERIES


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
73,TMPV,-8.70,-34.57,128.66,49.60,401411.0,-164878.0,311994.0,600.00,-87.07,37.0,X-LC,16.42,4.0,-0.41,2.04,0.46,XY24,NTT,AUTO
79,VALIANTORG,-4.60,-28.16,142.41,74.15,167386.0,-46067.0,117538.0,838.00,NaN,38.0,NaN,34.82,145.0,-0.28,0.77,74.91,XR,NTT,CHEMICALS
59,RELAXO,-4.52,-63.50,317.24,52.31,168099.0,-92172.0,52988.0,1176.00,NaN,43.0,NaN,13.14,92.0,-0.55,0.35,19.81,X40N,NTT,FOOTWEAR
69,TANLA,-4.33,-20.75,171.52,115.17,368025.0,-56194.0,214567.0,1903.93,NaN,71.0,NaN,21.18,147.0,-0.15,1.40,89.68,AR,ATH,IT
62,ROUTE,-3.64,-66.52,392.12,64.76,174693.0,-88515.0,44551.0,2170.75,NaN,47.0,NaN,11.22,141.0,-0.51,0.29,8.32,SR,ATH,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.19,58.94,29.89,106.44,70142.0,87022.0,234666.0,1629.94,15.89,51.0,M-SC,23.30,220.0,1.24,1.53,77.81,OX40N,ATH,CABLES
65,SIS,-1.82,-12.23,38.17,21.26,37196.0,-13584.0,97448.0,528.00,87.50,44.0,H-SC,6.33,168.0,-0.37,0.64,44.89,OX40N,NTT,MISC
12,BOSCH-HCIL,-3.18,-24.78,32.92,-0.01,31650.0,-31664.0,96141.0,2282.00,750.61,47.0,M-SC,19.43,231.0,-1.00,0.63,63.65,OX40N,BTT,AC
75,TTKPRESTIG,-0.28,-26.76,36.67,0.09,27064.0,-26972.0,73805.0,770.00,-10.92,55.0,M-SC,1.98,250.0,-1.00,0.48,32.84,OX40N,NTT,DURABLES
42,INDIGOPNTS,-1.65,-32.71,48.65,0.02,57141.0,-57106.0,117453.0,1408.00,-35.74,43.0,M-SC,7.53,222.0,-1.00,0.77,33.82,OX40N,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.19,58.94,29.89,106.44,70142.0,87022.0,234666.0,1629.94,15.89,51.0,M-SC,23.3,220.0,1.24,1.53,77.81,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.19,58.94,29.89,106.44,70142.0,87022.0,234666.0,1629.94,15.89,51.0,M-SC,23.3,220.0,1.24,1.53,77.81,OX40N,ATH,CABLES
66,STARHEALTH,-0.85,9.92,33.97,47.26,104974.0,27896.0,309019.0,761.00,-3.06,58.0,H-SC,4.0,173.0,0.27,2.02,33.61,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.19,58.94,29.89,106.44,70142.0,87022.0,234666.0,1629.94,15.89,51.0,M-SC,23.30,220.0,1.24,1.53,77.81,OX40N,ATH,CABLES
83,WIPRO,-0.54,-11.06,117.01,93.01,260430.0,-27669.0,222571.0,420.00,-25.10,59.0,M-LC,8.40,98.0,-0.11,1.45,23.44,XR,NTT,IT
69,TANLA,-4.33,-20.75,171.52,115.17,368025.0,-56194.0,214567.0,1903.93,NaN,71.0,NaN,21.18,147.0,-0.15,1.40,89.68,AR,ATH,IT
41,INDIAMART,-2.10,-32.34,196.94,100.92,164354.0,-39882.0,83454.0,4675.60,NaN,48.0,NaN,16.54,140.0,-0.24,0.54,5.88,AR,ATH,MISC
3,ALKYLAMINE,-1.39,-31.83,182.43,92.54,125567.0,-32133.0,68830.0,4520.85,NaN,39.0,NaN,7.79,142.0,-0.26,0.45,30.70,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,0.45,-17.93,61.88,32.85,136294.0,-48124.0,220255.0,50936.07,-39.04,31.0,X-MC,4.34,55.0,-0.35,1.44,5.18,X40,ATH,APPARELS
73,TMPV,-8.70,-34.57,128.66,49.60,401411.0,-164878.0,311994.0,600.00,-87.07,37.0,X-LC,16.42,4.0,-0.41,2.04,0.46,XY24,NTT,AUTO
6,AWL,0.74,-43.11,193.57,67.02,392392.0,-153596.0,202713.0,485.00,-58.01,38.0,X-SC,9.52,81.0,-0.39,1.32,0.83,XY24,NTT,FMCG
37,HONAUT,-1.07,-21.77,92.88,50.88,140109.0,-41988.0,150850.0,58192.80,-38.87,39.0,X-SC,8.37,90.0,-0.30,0.98,14.93,X40N,ATH,ELECTRICAL
55,PGHH,-2.32,-48.54,154.61,31.02,169873.0,-103648.0,109872.0,17484.40,-62.16,42.0,X-SC,22.93,57.0,-0.61,0.72,3.43,X40,ATH,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,-1.74,-12.85,32.37,15.37,50458.0,-22975.0,155880.0,34390.26,-29.03,48.0,X-MC,0.73,58.0,-0.46,1.02,5.59,X40,ATH,PHARMA
17,CIPLA,0.19,4.82,18.19,23.89,57327.0,14502.0,315157.0,1648.14,18.90,56.0,X-LC,1.03,18.0,0.25,2.06,18.05,X40N,ATH,PHARMA
76,UNITDSPR,2.06,6.22,13.69,20.77,34049.0,14568.0,248712.0,1644.00,-9.49,61.0,X-MC,1.88,64.0,0.43,1.62,19.55,X40N,NTT,BREWERIES
11,BERGEPAINT,-1.09,-13.33,44.59,25.32,99610.0,-34357.0,223392.0,680.00,-24.62,48.0,X-MC,2.31,68.0,-0.34,1.46,17.14,XY24,NTT,PAINTS
38,HYUNDAI,0.39,6.56,33.53,42.29,101321.0,18609.0,302180.0,2782.73,11.66,56.0,X-LC,2.65,29.0,0.18,1.97,24.39,X40N,ATH,AUTO


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
73,TMPV,-8.70,-34.57,128.66,49.60,401411.0,-164878.0,311994.0,600.00,-87.07,37.0,X-LC,16.42,4.0,-0.41,2.04,0.46,XY24,NTT,AUTO
6,AWL,0.74,-43.11,193.57,67.02,392392.0,-153596.0,202713.0,485.00,-58.01,38.0,X-SC,9.52,81.0,-0.39,1.32,0.83,XY24,NTT,FMCG
26,GILLETTE,-2.20,-16.26,53.67,28.69,114116.0,-41271.0,212625.0,10891.01,-39.51,43.0,X-SC,6.11,83.0,-0.36,1.39,2.73,X40,ATH,FMCG
55,PGHH,-2.32,-48.54,154.61,31.02,169873.0,-103648.0,109872.0,17484.40,-62.16,42.0,X-SC,22.93,57.0,-0.61,0.72,3.43,X40,ATH,FMCG
54,PAGEIND,0.45,-17.93,61.88,32.85,136294.0,-48124.0,220255.0,50936.07,-39.04,31.0,X-MC,4.34,55.0,-0.35,1.44,5.18,X40,ATH,APPARELS


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PGHH,-2.32,-48.54,154.61,31.02,169873.0,-103648.0,109872.0,17484.40,-62.16,42.0,X-SC,22.93,57.0,-0.61,0.72,3.43,X40,ATH,FMCG
2,ACC,0.23,-45.27,219.40,74.81,321965.0,-121380.0,146748.0,3906.00,-45.85,57.0,X-SC,12.89,85.0,-0.38,0.96,8.71,XY24,BTT,CEMENT
37,HONAUT,-1.07,-21.77,92.88,50.88,140109.0,-41988.0,150850.0,58192.80,-38.87,39.0,X-SC,8.37,90.0,-0.30,0.98,14.93,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-1.74,-12.85,32.37,15.37,50458.0,-22975.0,155880.0,34390.26,-29.03,48.0,X-MC,0.73,58.0,-0.46,1.02,5.59,X40,ATH,PHARMA
19,COLPAL,-0.60,-35.31,112.94,37.76,192427.0,-92985.0,170380.0,3628.04,-21.60,51.0,X-MC,14.67,62.0,-0.48,1.11,8.35,XY25,ATH,FMCG


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,3.54,-15.97,53.17,28.70,195503.0,-69900.0,367695.0,452.00,-24.24,73.0,X-LC,6.78,1.0,-0.36,2.40,15.73,X40,NTT,FMCG
71,TCS,-0.10,-27.57,74.98,26.75,265081.0,-134540.0,353535.0,4208.34,-41.29,50.0,X-LC,8.10,2.0,-0.51,2.31,22.62,X40,ATH,IT
44,INFY,0.32,-17.16,72.21,42.66,239794.0,-68776.0,332079.0,1972.00,-40.08,56.0,X-LC,7.20,3.0,-0.29,2.17,18.88,X40,NTT,IT
73,TMPV,-8.70,-34.57,128.66,49.60,401411.0,-164878.0,311994.0,600.00,-87.07,37.0,X-LC,16.42,4.0,-0.41,2.04,0.46,XY24,NTT,AUTO
33,HCLTECH,0.48,-5.26,32.68,25.70,111764.0,-18977.0,341996.0,1837.07,-9.66,53.0,X-LC,9.61,8.0,-0.17,2.23,36.45,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,-3.18,-24.78,32.92,-0.01,31650.0,-31664.0,96141.0,2282.00,750.61,47.0,M-SC,19.43,231.0,-1.00,0.63,63.65,OX40N,BTT,AC
79,VALIANTORG,-4.60,-28.16,142.41,74.15,167386.0,-46067.0,117538.0,838.00,NaN,38.0,NaN,34.82,145.0,-0.28,0.77,74.91,XR,NTT,CHEMICALS
27,GLAXO,-0.78,6.45,27.47,35.69,58274.0,12852.0,212136.0,3380.16,-20.11,54.0,X-MC,7.38,56.0,0.22,1.38,26.08,X40,ATH,PHARMA
69,TANLA,-4.33,-20.75,171.52,115.17,368025.0,-56194.0,214567.0,1903.93,NaN,71.0,NaN,21.18,147.0,-0.15,1.40,89.68,AR,ATH,IT
25,FINCABLES,-1.19,58.94,29.89,106.44,70142.0,87022.0,234666.0,1629.94,15.89,51.0,M-SC,23.30,220.0,1.24,1.53,77.81,OX40N,ATH,CABLES


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-4.33,-20.75,171.52,115.17,368025.0,-56194.0,214567.0,1903.93,NaN,71.0,NaN,21.18,147.0,-0.15,1.40,89.68,AR,ATH,IT
61,REPCOHOME,1.22,-16.91,128.45,89.82,305780.0,-48451.0,238054.0,880.00,NaN,72.0,NaN,10.01,161.0,-0.16,1.55,28.51,XY24,NTT,FINANCE
50,M&M,0.18,4.25,12.69,17.48,39830.0,12791.0,313866.0,3762.88,-8.98,68.0,X-LC,4.86,25.0,0.32,2.05,21.37,X40N,ATH,AUTO
34,HDFCBANK,1.55,2.50,31.41,34.70,116434.0,9039.0,370691.0,1073.00,-19.37,76.0,X-LC,6.73,26.0,0.08,2.42,20.04,X40,BTT,BANKS
81,VOLTAS,1.72,-9.83,61.03,45.20,105515.0,-18852.0,172890.0,1856.00,24.22,65.0,X-MC,18.15,79.0,-0.18,1.13,21.81,XY25,NTT,AC


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.93
1,25,51.68
2,50,82.44


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    43.17
SC    20.29
MC    19.75
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.37
XY24     22.70
X40N     19.30
XY25     10.00
XR        9.89
OX40N     6.72
AR        3.29
SR        0.74
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    41.72
X-MC    17.38
H-SC     8.23
X-SC     6.93
M-SC     5.13
M-LC     1.45
M-MC     1.07
H-MC     0.96
L-MC     0.34
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.46,-24.46,69.35
IT,12.75,-31.22,99.59
FINANCE,10.99,-21.02,72.53
AUTO,8.98,-16.14,58.81
MISC,5.59,-43.94,101.79
PAINTS,5.57,-24.01,46.29
INSURANCE,5.56,3.04,33.39
BANKS,5.28,-10.87,66.46
PHARMA,4.46,0.64,24.31


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4015623.0,19
X40,2140288.0,16
XR,1589280.0,13
X40N,1474588.0,12
AR,1003499.0,6
OX40N,760542.0,10
XY25,756258.0,6
SR,300260.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2941610.0,19
X-MC,1572843.0,13
X-SC,1213260.0,6
H-SC,889817.0,8
M-SC,378225.0,6
H-MC,266457.0,1
M-LC,260430.0,1
M-MC,81537.0,1
L-MC,50471.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1249148.0      8
X-SC       XY24       714357.0      2
X-LC       XY24       628746.0      2
           X40N       573201.0      5
H-SC       XY24       537149.0      3
X-MC       X40N       518374.0      4
           X40        485661.0      5
X-LC       XY25       376779.0      3
X-MC       XY25       297942.0      2
X-SC       X40        283989.0      2
X-MC       XY24       270866.0      2
H-MC       XY24       266457.0      1
M-LC       XR         260430.0      1
H-SC       OX40N      247375.0      3
X-SC       X40N       214914.0      2
M-SC       OX40N      185997.0      4
           XY24       131342.0      1
X-LC       XR         113736.0      1
H-SC       XR         105293.0      2
M-MC       XY25        81537.0      1
M-SC       XR          60886.0      1
L-MC       XR          50471.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 27.0 seconds
